# SingBERT vs HateXplain RoBERTa — Singlish Hate Speech Baseline Test

This notebook compares two models on Singlish hate speech detection — **without any SG-specific fine-tuning**:

1. **SingBERT** (`zanelim/singbert`) — knows Singlish, but never trained on hate speech
2. **HateXplain RoBERTa** (`shareeep/roberta-base-fine-tuned-hatexplain`) — trained on hate speech, but never seen Singlish

Both models output **3 classes**: `hate`, `offensive`, `normal` — enabling a direct comparison.

### Key Question
Which starting point is stronger for Singlish hate speech: **Singlish language knowledge** or **hate speech classification knowledge**?

---

## Cell 1 — Install Dependencies

In [ ]:
!pip install transformers torch scikit-learn --quiet

## Cell 2 — Load Models

- **SingBERT** — BERT pre-trained on Singlish/Manglish. No classifier head, so we add a random 3-class head.
- **HateXplain RoBERTa** — RoBERTa fine-tuned on HateXplain dataset with 3 classes (hate/offensive/normal). Already has a trained classification head.

In [ ]:
import torch
import numpy as np
from transformers import (
    BertTokenizer, BertForSequenceClassification,
    AutoTokenizer, AutoModelForSequenceClassification,
)

LABEL_MAP = {0: "hate", 1: "offensive", 2: "normal"}
LABEL_NAMES = ["hate", "offensive", "normal"]

# ── SingBERT (untrained classification head) ──────────────────────────────
singbert_tokenizer = BertTokenizer.from_pretrained('zanelim/singbert')
singbert_model = BertForSequenceClassification.from_pretrained(
    'zanelim/singbert',
    num_labels=3,
    id2label=LABEL_MAP,
    label2id={v: k for k, v in LABEL_MAP.items()},
)
singbert_model.eval()
print('SingBERT loaded (random classification head — not fine-tuned)')

# ── HateXplain RoBERTa (trained 3-class head) ────────────────────────────
hatex_tokenizer = AutoTokenizer.from_pretrained('shareeep/roberta-base-fine-tuned-hatexplain')
hatex_model = AutoModelForSequenceClassification.from_pretrained(
    'shareeep/roberta-base-fine-tuned-hatexplain',
)
hatex_model.eval()
print('HateXplain RoBERTa loaded (3-class: hate/offensive/normal)')
print(f'  Labels: {hatex_model.config.id2label}')

## Cell 3 — Helper Functions

In [ ]:
def classify(model, tokenizer, text, label_map=LABEL_MAP):
    """
    Run text through a model and return {label: probability} sorted by score.
    """
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=1).squeeze()
    scores = {label_map[i]: probs[i].item() for i in range(len(probs))}
    return dict(sorted(scores.items(), key=lambda x: x[1], reverse=True))


def singbert_classify(text):
    return classify(singbert_model, singbert_tokenizer, text)


def hatex_classify(text):
    return classify(hatex_model, hatex_tokenizer, text, hatex_model.config.id2label)


LABEL_EMOJI = {'hate': '\U0001f6a8', 'offensive': '\u26a0\ufe0f', 'normal': '\u2705'}


def run_inference(text):
    """Run both models on a text and print a formatted 3-class comparison."""
    print(f"\n{'='*65}")
    print(f'Input: "{text}"')
    print('='*65)

    # SingBERT
    sb_scores = singbert_classify(text)
    sb_top = list(sb_scores.keys())[0]
    print(f'\nSingBERT (untrained head):')
    for label, score in sb_scores.items():
        bar = chr(9608) * int(score * 30)
        print(f'  {LABEL_EMOJI.get(label, "  ")} {label:<12} {score:.3f}  {bar}')
    print(f'  -> Prediction: {LABEL_EMOJI.get(sb_top, "")} {sb_top.upper()}')

    # HateXplain RoBERTa
    hx_scores = hatex_classify(text)
    hx_top = list(hx_scores.keys())[0]
    print(f'\nHateXplain RoBERTa (trained 3-class):')
    for label, score in hx_scores.items():
        bar = chr(9608) * int(score * 30)
        print(f'  {LABEL_EMOJI.get(label, "  ")} {label:<12} {score:.3f}  {bar}')
    print(f'  -> Prediction: {LABEL_EMOJI.get(hx_top, "")} {hx_top.upper()}')

    # Agreement
    if sb_top == hx_top:
        print(f'\n  Models AGREE: {sb_top}')
    else:
        print(f'\n  Models DISAGREE: SingBERT={sb_top} vs HateXplain={hx_top}')
    print()


print('Helper functions ready.')

## Cell 4 — Interactive Test

Enter any Singlish (or English) phrase to see how both models classify it.

In [ ]:
user_input = input('Enter a sentence to classify (or press Enter for example): ').strip()

if not user_input:
    user_input = 'wah lao eh, this fella damn suay sia'
    print(f'Using example: "{user_input}"')

run_inference(user_input)

## Cell 5 — Batch Test: Singlish Examples

Test both models on hand-picked Singlish examples spanning all three categories.

> **Note:** Some examples contain offensive language for research/testing purposes only.

In [ ]:
# Test cases: (text, expected_label, notes)
test_cases = [
    # --- NORMAL ---
    ('wah lao eh, the mee pok today damn shiok leh',         'normal',     'casual food comment'),
    ('eh bro, later we go makan together can?',              'normal',     'friendly invite'),
    ('alamak, forgot to buy tissue again',                   'normal',     'mild frustration'),
    ('confirm plus chop, this show is sibei good',           'normal',     'positive recommendation'),
    ('ai yo, so leceh only',                                 'normal',     'mild complaint'),

    # --- OFFENSIVE ---
    ('nabeh, you damn stupid lah',                           'offensive',  'common Singlish expletive'),
    ('wah lan eh, you spoil my mood',                        'offensive',  'expletive, personal but not targeted hate'),
    ('cb, you sabo me again',                                'offensive',  'crude insult'),
    ('this fella bo liao one, sibei irritating',             'offensive',  'personal insult'),

    # --- HATE SPEECH ---
    ('those ah tiong always come here take our jobs',        'hate',       'xenophobia toward Chinese nationals'),
    ('all the bangla workers dirty and smelly one',          'hate',       'ethnic slur toward South Asian workers'),
    ('malays always so lazy, that\'s why poor',              'hate',       'racial stereotype'),
    ('india people cannot be trusted one',                   'hate',       'racial generalisation'),
]

print('Batch test on Singlish examples (3-class comparison)')
print(f"{'Text':<46} {'Expected':<11} {'SingBERT':<11} {'OK?':<4} {'HateXplain':<11} {'OK?':<4} Notes")
print('-' * 130)

sb_correct = 0
hx_correct = 0
total = len(test_cases)

for text, expected, notes in test_cases:
    sb_scores = singbert_classify(text)
    sb_pred = list(sb_scores.keys())[0]
    hx_scores = hatex_classify(text)
    hx_pred = list(hx_scores.keys())[0]

    sb_ok = sb_pred == expected
    hx_ok = hx_pred == expected
    sb_correct += sb_ok
    hx_correct += hx_ok

    print(
        f'{text[:45]:<46} {expected:<11} {sb_pred:<11} '
        f'{"Y" if sb_ok else "N":<4} {hx_pred:<11} '
        f'{"Y" if hx_ok else "N":<4} {notes}'
    )

print(f'\nSingBERT accuracy:         {sb_correct}/{total} ({sb_correct/total*100:.0f}%)')
print(f'HateXplain RoBERTa accuracy: {hx_correct}/{total} ({hx_correct/total*100:.0f}%)')
print('\n(SingBERT has a random head — low accuracy is expected.)')
print('(HateXplain was never trained on Singlish — Singlish-specific gaps are expected.)')

## Cell 6 — Batch Test on Synthetic Dataset

Evaluate both models on the actual synthetic test split (108 rows) for a more rigorous comparison.

In [ ]:
import json
from sklearn.metrics import classification_report, f1_score
from collections import Counter

SG_LABEL_TO_ID = {'hate': 0, 'offensive': 1, 'normal': 2}

# Load test split
test_path = '../data/singlish/synthetic/20_terms/split_60_20_20/test.jsonl'
# Adjust for Colab if needed
import os
if not os.path.exists(test_path):
    test_path = 'data/singlish/synthetic/20_terms/split_60_20_20/test.jsonl'

with open(test_path) as f:
    test_data = [json.loads(line) for line in f if line.strip()]

print(f'Loaded {len(test_data)} test examples')
print(f'Label distribution: {dict(Counter(d["label"] for d in test_data))}')

# Run both models
true_labels = [d['label'] for d in test_data]
true_ids = [SG_LABEL_TO_ID[l] for l in true_labels]

sb_preds = []
hx_preds = []

for d in test_data:
    sb_top = list(singbert_classify(d['text']).keys())[0]
    hx_top = list(hatex_classify(d['text']).keys())[0]
    sb_preds.append(SG_LABEL_TO_ID.get(sb_top, -1))
    hx_preds.append(SG_LABEL_TO_ID.get(hx_top, -1))

# SingBERT report
print('\n' + '='*60)
print('SingBERT (untrained) on SG Test Set')
print('='*60)
print(classification_report(true_ids, sb_preds, target_names=LABEL_NAMES))
sb_f1 = f1_score(true_ids, sb_preds, average='macro')
print(f'Macro F1: {sb_f1:.4f}')

# HateXplain report
print('\n' + '='*60)
print('HateXplain RoBERTa on SG Test Set')
print('='*60)
print(classification_report(true_ids, hx_preds, target_names=LABEL_NAMES))
hx_f1 = f1_score(true_ids, hx_preds, average='macro')
print(f'Macro F1: {hx_f1:.4f}')

## Cell 7 — Error Analysis: Where HateXplain Fails on Singlish

Identify which Singlish examples HateXplain gets wrong — these are the cases fine-tuning on SG data should fix.

In [ ]:
print('HateXplain RoBERTa — Misclassified Examples')
print(f"{'True':<11} {'Pred':<11} {'Term':<15} Text")
print('-' * 100)

errors_by_term = Counter()
errors_by_label = Counter()
n_errors = 0

for d, true_id, pred_id in zip(test_data, true_ids, hx_preds):
    if true_id != pred_id:
        n_errors += 1
        true_lbl = LABEL_NAMES[true_id]
        pred_lbl = LABEL_NAMES[pred_id]
        term = d.get('source_term', '?')
        errors_by_term[term] += 1
        errors_by_label[f'{true_lbl}->{pred_lbl}'] += 1
        if n_errors <= 20:  # show first 20
            print(f'{true_lbl:<11} {pred_lbl:<11} {term:<15} {d["text"][:80]}')

print(f'\nTotal errors: {n_errors}/{len(test_data)}')
print(f'\nErrors by confusion type: {dict(errors_by_label)}')
print(f'\nMost confused terms: {errors_by_term.most_common(10)}')

## Cell 8 — Tokenisation Comparison

Compare how both tokenizers handle Singlish words — SingBERT should know more Singlish vocabulary.

In [ ]:
singlish_words = [
    'lah', 'leh', 'lor', 'meh', 'sia',           # particles
    'wah', 'alamak', 'aiyo', 'shiok', 'bo liao',  # expressions
    'makan', 'lepak', 'sabo', 'kiasu', 'kena',    # common verbs/adj
    'ah tiong', 'bangla', 'ang moh',               # ethno-slang
    'nabeh', 'wah lan', 'cb',                      # expletives
    'ah kua', 'pukimak', 'chao',                   # from synthetic dataset
]

print('Tokenisation comparison — SingBERT vs HateXplain RoBERTa')
print(f'{"Word":<15} {"SingBERT tokens":<35} {"RoBERTa tokens":<35}')
print('-' * 85)

for word in singlish_words:
    sb_tokens = singbert_tokenizer.tokenize(word)
    hx_tokens = hatex_tokenizer.tokenize(word)
    print(f'{word:<15} {str(sb_tokens):<35} {str(hx_tokens):<35}')

print('\nFewer tokens = better vocabulary coverage for that word.')

## Cell 9 — Summary

Key takeaways from this baseline comparison.

In [ ]:
print(f"""
{'='*60}
BASELINE SUMMARY
{'='*60}

SingBERT (untrained head):    Macro F1 = {sb_f1:.4f}
HateXplain RoBERTa:           Macro F1 = {hx_f1:.4f}

SingBERT has Singlish language understanding but no hate
speech training — its classification head is random.

HateXplain RoBERTa knows hate speech patterns but has
never seen Singlish — it misses culture-specific slurs.

NEXT: Fine-tuning experiments (singbert_sg_experiments.ipynb)
  Exp A:  SingBERT -> SG only
  Exp B:  SingBERT -> HateXplain -> SG (two-stage)
  Exp C1: HateXplain RoBERTa -> SG
  Exp C2: Combined training

These will show which starting point benefits more from
SG fine-tuning, and whether combining both knowledge
sources outperforms either alone.
{'='*60}
""")